In [1]:
import pandas as pd
import pickle

In [2]:
cna_data = pd.read_csv("filtered_cna_data.csv")
cna_data = cna_data.rename(columns={"cna_data$Entrez_Gene_Id": "Entrez_Gene_Id"})
cna_data.head()

,Entrez_Gene_Id,C3L-00017,C3L-00102,C3L-00277,C3L-00589,C3L-00598,C3L-00599,C3L-00622,C3L-00625,C3L-00819,...,C3N-03665,C3N-03666,C3N-03754,C3N-03839,C3N-03840,C3N-03853,C3N-03884,C3N-04119,C3N-04126,C3N-04282
0,1,0,0,0,0,1,0,0,0,0,...,0,1,1,1,0,1,0,0,0,0
1,10,-1,0,-1,0,1,0,0,0,0,...,0,1,0,0,0,0,0,-1,-1,-1
2,100,0,0,0,0,1,0,0,0,0,...,0,1,0,1,0,1,0,0,0,0
3,1000,0,0,1,1,0,0,0,0,0,...,1,0,1,0,0,2,0,0,-1,-1
4,10000,1,0,0,0,1,0,0,0,0,...,1,1,1,1,0,1,0,0,1,0


In [3]:
cnv_entrezid = pd.read_excel("cna_entrezid.xlsx")
cnv_entrezid

,Cytoband,ensembl_gene_id,hgnc_symbol,chromosome_name,start_position,end_position,strand,description,entrezgene_id
0,9p21.3,ENSG00000264545,NaN,9,21802636,22029594,1,novel transcript,NaN
1,9p21.3,ENSG00000224854,CDKN2A-AS1,9,21959833,21967755,1,CDKN2A antisense RNA 1 [Source:HGNC Symbol;Acc...,NaN
2,9p21.3,ENSG00000147889,CDKN2A,9,21967752,21995301,-1,cyclin dependent kinase inhibitor 2A [Source:H...,1029.0
3,17p12,ENSG00000007174,DNAH9,17,11598470,11969748,1,dynein axonemal heavy chain 9 [Source:HGNC Sym...,1770.0
4,17p12,ENSG00000266368,NaN,17,11953889,11977594,-1,"novel transcript, antisense to DNAH9",NaN
...,...,...,...,...,...,...,...,...,...
1108,21q11.2,ENSG00000307760,NaN,21,15508837,15566124,1,novel transcript,NaN
1109,21q11.2,ENSG00000212564,RNU6-1326P,21,15614283,15614389,1,"RNA, U6 small nuclear 1326, pseudogene [Source...",NaN
1110,21q11.2,ENSG00000226298,RAD23BP3,21,15694300,15696581,-1,RAD23B pseudogene 3 [Source:HGNC Symbol;Acc:HG...,NaN
1111,21q11.2,ENSG00000155313,USP25,21,15729925,15880064,1,ubiquitin specific peptidase 25 [Source:HGNC S...,29761.0


In [4]:
my_dict = {}
for cytoband in cnv_entrezid["Cytoband"].unique():
    filtered_cnv_entrezid = cnv_entrezid[cnv_entrezid["Cytoband"] == cytoband].dropna(subset="entrezgene_id")
    filtered_cnv_entrezid["entrezgene_id"] = filtered_cnv_entrezid["entrezgene_id"].astype(int)
    for _,row in filtered_cnv_entrezid.iterrows():
        my_dict[row["entrezgene_id"]] = row["Cytoband"]
cna_data = cna_data[cna_data["Entrez_Gene_Id"].isin(cnv_entrezid["entrezgene_id"].unique())]
cna_data["Cytoband"] = cna_data["Entrez_Gene_Id"].apply(lambda x: my_dict[x])
cna_data.head()

,Entrez_Gene_Id,C3L-00017,C3L-00102,C3L-00277,C3L-00589,C3L-00598,C3L-00599,C3L-00622,C3L-00625,C3L-00819,...,C3N-03666,C3N-03754,C3N-03839,C3N-03840,C3N-03853,C3N-03884,C3N-04119,C3N-04126,C3N-04282,Cytoband
1179,100132288,0,0,-1,0,1,0,0,0,0,...,-2,0,1,0,1,0,1,0,0,21q11.2
3695,100423008,0,0,0,0,1,0,0,0,0,...,0,-1,1,0,1,0,0,-1,-1,21q11.2
3704,100423018,0,0,0,0,1,0,0,0,0,...,0,-1,1,0,1,0,1,-1,-1,21q11.2
6110,100874190,0,0,0,0,1,0,0,0,0,...,0,-1,1,0,1,0,0,-1,-1,21q11.2
8388,1029,-1,0,-2,0,-2,0,0,-1,-1,...,-2,-1,0,0,-2,-1,-1,-2,-1,9p21.3


In [5]:
X_mean = cna_data.drop(columns="Entrez_Gene_Id").groupby("Cytoband").mean().round().T[['21q11.2', '17p12', '18q21.2', '9p21.3']].clip(upper=0)
X_mean.to_csv("X_mean.csv")
X_mean

Cytoband,21q11.2,17p12,18q21.2,9p21.3
C3L-00017,0.0,-1.0,0.0,-1.0
C3L-00102,0.0,0.0,0.0,0.0
C3L-00277,-0.0,0.0,-1.0,-2.0
C3L-00589,0.0,0.0,0.0,0.0
C3L-00598,0.0,0.0,0.0,-2.0
...,...,...,...,...
C3N-03853,0.0,0.0,0.0,-2.0
C3N-03884,0.0,-1.0,0.0,-1.0
C3N-04119,0.0,-1.0,-1.0,-1.0
C3N-04126,-1.0,-1.0,-1.0,-2.0


In [6]:
band_values = {}
for col in X_mean.columns:
    cytoband = cna_data[cna_data["Cytoband"] == col].drop(columns="Entrez_Gene_Id")
    band_values[col] = [cytoband[pat].eq(X_mean.loc[pat, col]).mean() for pat in X_mean.index]
pd.DataFrame(band_values).mean().round(2)

21q11.2    0.88
17p12      0.98
18q21.2    0.95
9p21.3     0.98
dtype: float64

In [9]:
with open("cna_rfmodel.pkl", "rb") as f:
    model = pickle.load(f)

C:\Users\Carlos Paja Suarez\anaconda3\envs\imoc-pdac\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Carlos Paja Suarez\anaconda3\envs\imoc-pdac\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [10]:
pd.Series(model.predict(X_mean), index=X_mean.index).to_csv("clusters_mean.csv")